In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_mmda_traffic_spatial.csv', parse_dates=['Date'])
df.info()
df.describe()
df.isnull().sum()

In [ ]:
df.duplicated().sum()

# Check for duplicates

In [ ]:
duplicates = df[df.duplicated(keep=False)]
duplicates

# Check for duplicate rows

In [ ]:
df["Date"].isna().sum()

# Check for invalid dates


In [ ]:
invalid_coordinates = df[
    (df["Latitude"] < -90) |
    (df["Latitude"] > 90) |
    (df["Longitude"] < -180) |
    (df["Longitude"] > 180)
]

invalid_coordinates

In [ ]:
df[
    (df["Latitude"] < 14) |
    (df["Latitude"] > 15) |
    (df["Longitude"] < 120) |
    (df["Longitude"] > 122)
]

In [ ]:
invalid_coordinates = (
    (df["Latitude"] == 0) |
    (df["Longitude"] == 0)
)

print("Invalid coordinate records:", invalid_coordinates.sum())

In [ ]:
df.loc[
    invalid_coordinates,
    ["Latitude", "Longitude"]
] = np.nan

In [ ]:
df[["Latitude", "Longitude"]].isnull().sum()

In [ ]:
df.loc[
    df[["Latitude", "Longitude"]].isna().any(axis=1),
    ["Date", "City", "Location", "Latitude", "Longitude", "Type"]
]

In [ ]:
parsed_time = pd.to_datetime(
    df["Time"],
    format="mixed",
    errors="coerce"
)

invalid_times = df[
    df["Time"].notna() & parsed_time.isna()
]

invalid_times[["Date", "Time"]]

In [ ]:
pd.set_option("display.max_colwidth", None)

print(
    df.loc[
        invalid_times.index,
        ["Date", "Time", "Tweet"]
    ].to_string()
)

In [ ]:
time_corrections = {
    4081: "6:20 AM",
    4340: "6:58 AM",
    5668: "7:50 AM",
    5690: "9:18 AM",
    5698: "1:15 PM",
    9617: "7:21 AM",
    9618: "7:57 AM",
    9619: "8:26 AM",
    10471: "12:04 PM",
    13547: "10:55 PM"
}

for index, corrected_time in time_corrections.items():
    df.loc[index, "Time"] = corrected_time

In [ ]:
parsed_time = pd.to_datetime(
    df["Time"],
    format="mixed",
    errors="coerce"
)

invalid_times = df[
    df["Time"].notna() & parsed_time.isna()
]

invalid_times[["Date", "Time"]]

In [ ]:
print("Negative lanes blocked:")
print(df[df["Lanes_Blocked"] < 0])

print("Invalid High_Accuracy values:")
print(df[~df["High_Accuracy"].isin([0, 1])])

In [ ]:
df["Lanes_Blocked"].describe()

In [ ]:
text_columns = ["City", "Location", "Direction", "Type", "Involved"]

for column in text_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False).head(20))

In [ ]:
df["City"] = df["City"].astype("string").str.strip()

df["City"] = df["City"].replace(
    "ParaÃ±aque",
    "Parañaque"
)

In [ ]:
df["City"].value_counts(dropna=False)

In [ ]:
for column in ["Direction", "Type", "Involved"]:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False).head(30))

In [ ]:
for column in ["Direction", "Type", "Involved"]:
    print(
        column,
        df[column].astype("string").str.strip().ne(df[column]).sum()
    )

In [ ]:
valid_directions = ["NB", "SB", "EB", "WB"]

direction_issues = df[
    df["Direction"].notna() &
    ~df["Direction"].isin(valid_directions)
]

direction_issues[
    ["Date", "Time", "City", "Location", "Direction", "Type", "Tweet"]
]

In [ ]:
df.loc[df["Direction"] == "EB.", "Direction"] = "EB"

invalid_direction_values = ["DAR", "CLOSED", "PAX", "CLARA"]

df.loc[
    df["Direction"].isin(invalid_direction_values),
    "Direction"
] = np.nan

In [ ]:
df["Direction"].value_counts(dropna=False)

In [ ]:
print("Type values:")
print(df["Type"].value_counts(dropna=False).head(30))

print("\nInvolved values:")
print(df["Involved"].value_counts(dropna=False).head(30))

In [ ]:
df["Involved"].value_counts(dropna=False).head(30)

In [ ]:
for value in df["Involved"].dropna().unique():
    if value.strip() != value or "  " in value:
        print(repr(value))

In [ ]:
df["Involved"].dropna().astype("string").str.strip().value_counts().head(30)

In [ ]:
df["Involved"] = (
    df["Involved"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [ ]:
for value in df["Involved"].dropna().unique():
    if value.strip() != value or "  " in value:
        print(repr(value))

In [ ]:
df["Involved"].value_counts(dropna=False).head(30)

In [ ]:
involved_is_tweet = (
    df["Involved"]
    .astype("string")
    .str.contains("MMDA ALERT", case=False, na=False)
)

print("Invalid Involved values:", involved_is_tweet.sum())

In [ ]:
df.loc[
    involved_is_tweet,
    ["Date", "Time", "City", "Location", "Involved", "Tweet"]
]

In [ ]:
df.loc[involved_is_tweet, "Involved"] = pd.NA

In [ ]:
df.loc[
    involved_is_tweet,
    ["Date", "Time", "City", "Location", "Involved", "Tweet"]
]

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)

In [ ]:
df.to_csv(
    "data_mmda_traffic_cleaned.csv",
    index=False
)